In [1]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

focal_df = pd.read_csv(
    PROCESSED_DIR / "focal_2016_2025.csv"
)

dev_df = pd.read_csv(
    PROCESSED_DIR / "development_2016_2025.csv"
)

print("Focal:", focal_df.shape)
print("Development:", dev_df.shape)

Focal: (180, 10)
Development: (1755, 8)


In [3]:
print("Focal columns:")
print(focal_df.columns.tolist())

print("\nDevelopment columns:")
print(dev_df.columns.tolist())

Focal columns:
['corpus', 'iso3', 'session', 'year', 'filename', 'filepath', 'text', 'word_count', 'country', 'region']

Development columns:
['corpus', 'iso3', 'session', 'year', 'filename', 'filepath', 'text', 'word_count']


In [4]:
print("Missing focal text:", focal_df["text"].isna().sum())
print("Missing development text:", dev_df["text"].isna().sum())

print("Empty focal text:",
      (focal_df["text"].fillna("").str.strip() == "").sum())

print("Empty development text:",
      (dev_df["text"].fillna("").str.strip() == "").sum())

Missing focal text: 0
Missing development text: 0
Empty focal text: 0
Empty development text: 0


In [5]:
focal_df["text_clean"] = focal_df["text"].copy()
dev_df["text_clean"] = dev_df["text"].copy()

In [6]:
def normalize_unicode_and_whitespace(text):
    if pd.isna(text):
        return ""

    # Normalize Unicode without removing meaningful characters
    text = unicodedata.normalize("NFC", text)

    # Replace unusual Unicode line/paragraph separators
    text = text.replace("\u2028", "\n")
    text = text.replace("\u2029", "\n")

    # Replace non-breaking spaces with normal spaces
    text = text.replace("\u00A0", " ")

    # Standardize different newline formats
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Convert tabs to spaces
    text = text.replace("\t", " ")

    # Collapse repeated spaces within lines
    text = re.sub(r"[ ]{2,}", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [7]:
focal_df["text_clean"] = focal_df["text"].apply(
    normalize_unicode_and_whitespace
)

dev_df["text_clean"] = dev_df["text"].apply(
    normalize_unicode_and_whitespace
)

In [8]:
print("Empty focal cleaned text:",
      (focal_df["text_clean"].str.strip() == "").sum())

print("Empty development cleaned text:",
      (dev_df["text_clean"].str.strip() == "").sum())

Empty focal cleaned text: 0
Empty development cleaned text: 0


In [9]:
focal_df["original_chars"] = focal_df["text"].str.len()
focal_df["clean_chars"] = focal_df["text_clean"].str.len()

focal_df[
    ["iso3", "year", "original_chars", "clean_chars"]
].head(10)

,iso3,year,original_chars,clean_chars
0,CHL,2016,8039,8038
1,CHL,2017,15549,15549
2,CHL,2018,26641,26558
3,CHL,2019,16335,16334
4,CHL,2020,19274,19274
5,CHL,2021,18698,18697
6,CHL,2022,14946,14945
7,CHL,2023,17065,17064
8,CHL,2024,12437,12435
9,CHL,2025,16946,16946


In [10]:
print("ORIGINAL:")
print(focal_df.loc[0, "text"][:1500])

print("\n" + "=" * 80 + "\n")

print("CLEANED:")
print(focal_df.loc[0, "text_clean"][:1500])

ORIGINAL:
We are living in times that show us the necessity of shifting our perspective with regard to addressing development. Our world is in the midst of a crisis that extends back years, and which, far from being resolved, has only worsened, generating criticism of political institutions that do not seem capable of representing their citizens. At the same time, economic growth has given way to a global slowdown and has not been able to provide the world’s population with the well-being that it seeks.
It would appear that the far-reaching dynamics that caused the crisis some years ago were not superseded but merely superficially retouched. But citizens today are more aware and empowered. Thanks to new technology, they have new means of expression. They are showing us that in their daily lives, at their places of work, at their children’s schools, in the health of their families, in the security of their cities, in the pensions of their mothers and fathers and in the relations between

In [11]:
focal_df["text_lower"] = focal_df["text_clean"].str.lower()
dev_df["text_lower"] = dev_df["text_clean"].str.lower()

In [12]:
focal_df["word_count_clean"] = (
    focal_df["text_clean"]
    .str.split()
    .str.len()
)

dev_df["word_count_clean"] = (
    dev_df["text_clean"]
    .str.split()
    .str.len()
)

In [13]:
focal_df["word_count_clean"].describe()

count     180.000000
mean     2171.383333
std       951.973952
min       611.000000
25%      1554.000000
50%      1978.500000
75%      2571.750000
max      8242.000000
Name: word_count_clean, dtype: float64

In [14]:
dev_df["word_count_clean"].describe()

count    1755.000000
mean     2037.878063
std       702.201454
min       463.000000
25%      1580.500000
50%      1939.000000
75%      2398.000000
max      7082.000000
Name: word_count_clean, dtype: float64

In [15]:
print(
    "Total focal words:",
    focal_df["word_count_clean"].sum()
)

print(
    "Total development words:",
    dev_df["word_count_clean"].sum()
)

Total focal words: 390849
Total development words: 3576476


In [16]:
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
focal_df.to_csv(
    PREPROCESSED_DIR / "focal_preprocessed.csv",
    index=False
)

dev_df.to_csv(
    PREPROCESSED_DIR / "development_preprocessed.csv",
    index=False
)

In [18]:
focal_df[
    ["iso3", "year", "filename", "word_count_clean"]
].sort_values("word_count_clean").head(5)

,iso3,year,filename,word_count_clean
55,IDN,2021,IDN_76_2021.txt,611
74,JOR,2020,JOR_75_2020.txt,738
75,JOR,2021,JOR_76_2021.txt,903
54,IDN,2020,IDN_75_2020.txt,950
174,USA,2020,USA_75_2020.txt,981


In [19]:
dev_df[
    ["iso3", "year", "filename", "word_count_clean"]
].sort_values("word_count_clean").head(5)

,iso3,year,filename,word_count_clean
13,BEN,2016,BEN_71_2016.txt,463
945,GNB,2021,GNB_76_2021.txt,480
1017,RWA,2021,RWA_76_2021.txt,492
447,LTU,2018,LTU_73_2018.txt,508
269,LTU,2017,LTU_72_2017.txt,537


In [20]:
shortest_idx = focal_df["word_count_clean"].idxmin()

print(
    focal_df.loc[
        shortest_idx,
        ["iso3", "year", "filename", "word_count_clean"]
    ]
)

print("\n--- BEGINNING OF SPEECH ---\n")
print(focal_df.loc[shortest_idx, "text_clean"][:2000])

iso3                            IDN
year                           2021
filename            IDN_76_2021.txt
word_count_clean                611
Name: 55, dtype: object

--- BEGINNING OF SPEECH ---

His Excellency President of the UN General Assembly,
His Excellency UN Secretary-General,
Excellencies, Leaders of UN Member Countries.
The world community looks towards the UN General Assembly to provide answers to major global concerns.
When will the people be free from the pandemic? When will the economy recover and grow inclusively? How to sustain the planet for future generations? And when will the world be free from conflict, terrorism, and war?
Given the recent global developments, there are many things that we need to do together.
First, we must bring hope that we can tackle the COVID-19 pandemic swiftly, fairly, and equally.
We know that “no one is safe until everyone is”. The capacity and pace among countries in tackling COVID-19, including in vaccinations, differs widely. Politici